# FLAG 2027 - Notebook 21 [v2 27-Oct]: two multilingual voice encoders (stage 1: challenge + all MAV, no VoxCeleb)

| Name | Model | Why |
|---|---|---|
| `sam100` | WeSpeaker SimAM-ResNet100 (VoxBlink2 pretrain -> VoxCeleb2 fine-tune), PyTorch checkpoint `voxblink2_samresnet100_ft.zip`, 256-d | Vox1-O 0.229% EER (as good as ReDimNet2), 50M params, fast |
| `w2vb` | WeSpeaker W2V-BERT 2.0 + MFA head (fine-tuned LM checkpoint, 600M), 256-d | best published CROSS-LINGUAL speaker evidence (3.1% vs 9.1% on unseen languages) -> aimed at our Bangla cells |

Both are loaded through WeSpeaker's own loader (`wespeaker.cli.speaker.Speaker`) and used strictly frozen (allowed by the organizers' FAQ even though their pretraining corpora may contain Bengali); both are declared in the system description.

v2 changes: no ONNX (the SimAM ONNX graph has a dynamic Clip that onnx2torch cannot convert); torchaudio backend shim; config patching for the base-model path; checkpoint key-match report; a speaker-discrimination sanity test on 120 training clips before each long extraction.

Steps: file lists in our row order (alignment check) -> extract both encoders -> compare combos on top of R4 (reference recomputed, already cached from v1) -> ZIP of the best (no VoxCeleb). If a combo wins by >= 0.5 on both std and gender, stage 2 adds VoxCeleb (re-download) and retrains system A.

About 3-4 h (w2vb is slow). Resumable: S0 results are cached in results/nb21_cv.csv from the first run. Paste the **SUMMARY**.

In [ ]:
# Cell 1 [v2 27-Oct] - Drive, GPU, packages. Both encoders are loaded through WeSpeaker's own loader (no ONNX any more).
from google.colab import drive
drive.mount('/content/drive')
!pip -q install speechbrain pyyaml peft
!pip -q uninstall -y torchao
import subprocess, sys, os
WESPEAKER_OK = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/wenet-e2e/wespeaker.git']).returncode == 0
if not WESPEAKER_OK:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/wenet-e2e/wespeaker.git', '/content/wespeaker_src'])
    sys.path.insert(0, '/content/wespeaker_src')
import torch, shutil, torchaudio
# torchaudio >= 2.9 removed the backend setters that WeSpeaker still calls -> harmless no-op shims
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda *a, **k: None
if not hasattr(torchaudio, 'get_audio_backend'):
    torchaudio.get_audio_backend = lambda *a, **k: 'soundfile'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('[v2 27-Oct] wespeaker pip install ok:', WESPEAKER_OK, '| Device:', dev, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - file lists in the SAME row order as our feature files (challenge, MAV v1/v3, MAV extra) + alignment check
import re, json, time, zipfile, copy
import numpy as np, pandas as pd
import librosa, soundfile as sf
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL, members=[m for m in zf.namelist() if m.endswith(('.wav', '.txt', '.csv'))])
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FILES = {}
lp = find_one('train_English.txt'); FILES['train'] = [resolve(l.split()[2], lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
    FILES[s] = [resolve(l.split()[1], lp) for l in open(lp) if l.strip()]
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_paths, mspk = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver))
            mav_paths.append(os.path.join(MAVD, ver, vm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FILES['mav'] = mav_paths
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            _ = fr[rng.integers(len(fr))]
            if need:
                zf.extract(vn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
FILES['extra'] = [r[1] for r, k in zip(rows, keep) if k]
# alignment check with the official ECAPA (100 rows per set)
try:
    from speechbrain.inference.interfaces import Pretrained
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
rng = np.random.default_rng(1)
for k, saved in {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}.items():
    idx = rng.choice(len(saved), 100, replace=False)
    new = np.stack([ECAPA.encode_batch(torch.from_numpy(librosa.load(FILES[k][i], sr=16000, mono=True)[0]).unsqueeze(0)).squeeze().cpu().numpy() for i in idx]); old = saved[idx]
    cos = (new * old).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(old, axis=1) + 1e-9)
    note('alignment %-6s median cosine %.4f' % (k, np.median(cos))); assert np.median(cos) > 0.98
note('Files: ' + ', '.join('%s %d' % (k, len(v)) for k, v in FILES.items()))

In [ ]:
# Cell 3 [v4 27-Oct] - our OWN minimal loader for the two WeSpeaker checkpoints (only the model files are imported; no WeSpeaker CLI, no s3prl)
import requests, tarfile, yaml, importlib.util, types, inspect
import torchaudio
def modelscope_url(fname):
    tree = requests.get('https://modelscope.cn/api/v1/datasets/wenet/wespeaker_pretrained_models/oss/tree', timeout=60).json()
    for it in tree['Data']:
        if it['Key'].endswith(fname):
            return it['Url']
    raise RuntimeError('file not found in modelscope listing: ' + fname)
def download(fname, dest):
    if os.path.exists(dest):
        return dest
    url = modelscope_url(fname)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status(); tot = int(r.headers.get('content-length', 0))
        with open(dest + '.part', 'wb') as f, tqdm(total=tot, unit='B', unit_scale=True, desc=fname) as bar:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk); bar.update(len(chunk))
    os.rename(dest + '.part', dest); return dest
WDIR = os.path.join(CACHE, 'wespeaker'); os.makedirs(WDIR, exist_ok=True)

# Register the wespeaker packages WITHOUT running their __init__.py files (they import the CLI and s3prl, which break),
# then import only the three plain-PyTorch files we need.
for k in [k for k in sys.modules if k == 'wespeaker' or k.startswith('wespeaker.')]:
    del sys.modules[k]
_spec = importlib.util.find_spec('wespeaker'); assert _spec is not None, 'wespeaker not installed - rerun Cell 1'
WROOT = os.path.dirname(_spec.origin)
for pkg, sub in [('wespeaker', ''), ('wespeaker.models', 'models'), ('wespeaker.frontend', 'frontend')]:
    _m = types.ModuleType(pkg); _m.__path__ = [os.path.join(WROOT, sub)]; sys.modules[pkg] = _m
import wespeaker.models.samresnet as samresnet
import wespeaker.models.w2vbert_adapter_mfa as w2vmfa
import wespeaker.frontend.w2vbert as w2vfe
print('[v4 27-Oct] wespeaker model files imported from', WROOT)

def unpack(fname):
    arc = download(fname, os.path.join(WDIR, fname)); root = '/content/wesp_' + fname.split('.')[0]
    if not os.path.exists(os.path.join(root, '.done')):
        (zipfile.ZipFile(arc) if fname.endswith('.zip') else tarfile.open(arc)).extractall(root)
        open(os.path.join(root, '.done'), 'w').close()
    mdir = [r for r, d, f in os.walk(root) if 'config.yaml' in f][0]
    pts = [f for f in os.listdir(mdir) if f.endswith('.pt')]
    assert pts, 'no .pt file next to config.yaml in ' + mdir
    pt = os.path.join(mdir, 'avg_model.pt' if 'avg_model.pt' in pts else pts[0])
    return mdir, pt
def kw_filter(cls, args):
    sig = inspect.signature(cls.__init__).parameters
    dropped = [k for k in args if k not in sig]
    return {k: v for k, v in args.items() if k in sig}, dropped
def build(mdir, flip_lora=False):
    cfg = yaml.safe_load(open(os.path.join(mdir, 'config.yaml'))); name = cfg['model']; notes = []
    if name.startswith('SimAM_ResNet'):
        cls = getattr(samresnet, name)
    elif name.startswith('W2VBert_Adapter_MFA'):
        cls = getattr(w2vmfa, name)
    else:
        raise RuntimeError('unexpected model type in config: ' + name)
    margs, dropped = kw_filter(cls, dict(cfg.get('model_args', {})))
    if dropped:
        notes.append('model_args dropped %s' % dropped)
    model = cls(**margs)
    da = cfg.get('dataset_args', {}); ft = da.get('frontend', 'fbank')
    if ft != 'fbank':
        fa = dict(da.get(ft + '_args', {}))
        if isinstance(fa.get('model_name'), str) and not os.path.isdir(fa['model_name']) and fa['model_name'] != 'facebook/w2v-bert-2.0':
            notes.append('model_name %s -> facebook/w2v-bert-2.0' % fa['model_name']); fa['model_name'] = 'facebook/w2v-bert-2.0'
        if fa.get('use_bnb'):
            notes.append('use_bnb -> False'); fa['use_bnb'] = False
        if flip_lora and 'use_lora' in fa:
            fa['use_lora'] = not fa['use_lora']; notes.append('use_lora -> %s' % fa['use_lora'])
        fa['sample_rate'] = da.get('resample_rate', 16000)
        fa, dropped = kw_filter(w2vfe.W2VBertFrontend, fa)
        if dropped:
            notes.append('frontend_args dropped %s' % dropped)
        model.add_module('frontend', w2vfe.W2VBertFrontend(**fa))
    model.frontend_type = ft; model.cfg_name = name; model.lora_flag = (ft != 'fbank') and 'use_lora' in da.get(ft + '_args', {})
    return model, notes
def load_weights(model, pt):
    try:
        ck = torch.load(pt, map_location='cpu')
    except Exception:
        ck = torch.load(pt, map_location='cpu', weights_only=False)
    for key in ('state_dict', 'model'):
        if isinstance(ck, dict) and key in ck and isinstance(ck[key], dict):
            ck = ck[key]
    res = model.load_state_dict(ck, strict=False)
    missing = [k for k in res.missing_keys if not k.endswith('num_batches_tracked')]
    return missing, list(res.unexpected_keys)
def load_wespeaker(fname):
    mdir, pt = unpack(fname)
    model, notes = build(mdir); missing, unexpected = load_weights(model, pt)
    if (missing or unexpected) and model.lora_flag:            # config/checkpoint disagree on LoRA wrapping -> try the other way
        m2, n2 = build(mdir, flip_lora=True); mi2, un2 = load_weights(m2, pt)
        if len(mi2) + len(un2) < len(missing) + len(unexpected):
            model, notes, missing, unexpected = m2, n2, mi2, un2
    note('%s: %s (frontend=%s) | config notes: %s | checkpoint keys missing %d / unexpected %d' % (fname.split('.')[0], model.cfg_name, model.frontend_type, notes or 'none', len(missing), len(unexpected)))
    if missing:
        print('  first missing keys:', missing[:5])
    if unexpected:
        print('  first unexpected keys:', unexpected[:5])
    assert len(missing) < 20, 'too many missing weights - the checkpoint does not match the model, tell Claude'
    return model.to(dev).eval()
@torch.no_grad()
def embed(model, pcm):
    """pcm: [1, T] float tensor. fbank models: WeSpeaker CLI reads wavs with normalize=False (int16 scale) -> caller passes x32768."""
    if model.frontend_type == 'fbank':
        feat = torchaudio.compliance.kaldi.fbank(pcm, num_mel_bins=80, frame_length=25, frame_shift=10, sample_frequency=16000, window_type='hamming')
        feat = (feat - feat.mean(0, keepdim=True)).unsqueeze(0).to(dev)      # [1, T, 80], per-utterance CMN as in the CLI
        out = model(feat)
    else:
        wav = pcm.to(dev); states, _ = model.frontend(wav, torch.LongTensor([wav.shape[1]]).to(dev)); out = model(states)
    out = out[-1] if isinstance(out, (tuple, list)) else out
    return out[0].detach().float().cpu().numpy().reshape(-1)

def _l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def sanity(name, one):
    """120 random TRAIN clips: same-identity cosine must clearly exceed different-identity cosine, else the loader is wrong."""
    rng = np.random.default_rng(7); idx = rng.choice(len(FILES['train']), 120, replace=False)
    E = _l2(np.stack([one(FILES['train'][i]) for i in idx])); lab = y[idx].astype(int); C = E @ E.T
    same = (lab[:, None] == lab[None, :]) & ~np.eye(len(idx), dtype=bool); diff = lab[:, None] != lab[None, :]
    gap = C[same].mean() - C[diff].mean()
    note('%s sanity: same-id cosine %.3f | different-id %.3f | gap %.3f (%d same pairs)' % (name, C[same].mean(), C[diff].mean(), gap, same.sum() // 2))
    assert gap > 0.10, 'embeddings are not speaker-discriminative -> loader/feature mismatch, tell Claude'
def make_encoder(fname, sec, int16_scale, autocast):
    def fn(paths):
        m = load_wespeaker(fname); L = 16000 * sec
        def one(p):
            w = librosa.load(p, sr=16000, mono=True)[0]
            w = w[(len(w) - L) // 2:(len(w) - L) // 2 + L] if len(w) >= L else np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]
            pcm = torch.from_numpy(w * (32768.0 if int16_scale else 1.0)).float().unsqueeze(0)   # [1, T]
            with torch.autocast('cuda', dtype=torch.float16, enabled=(autocast and dev == 'cuda')):
                return embed(m, pcm)
        sanity(fname.split('.')[0], one)
        out = np.stack([one(p) for p in tqdm(paths, desc=fname.split('.')[0])])
        del m; torch.cuda.empty_cache(); return out
    return fn
# sam100: fbank model, 6-s centre crop (repeat-pad), int16 scale.   w2vb: raw waveform in [-1, 1], 8-s crop, fp16 autocast for the 600M conformer.
ENCODERS = {'sam100': make_encoder('voxblink2_samresnet100_ft.zip', 6, True, False),
            'w2vb': make_encoder('voxceleb_voxblink2_w2v_bert2_lora_adapterMFA_lm.tar.gz', 8, False, True)}
print('[v4 27-Oct] encoders defined')

In [ ]:
# Cell 4 - extract + save (sam100 first: fast; w2vb second: slow). Skipped if present.
ORDER = SPLIT_NAMES + ['mav', 'extra']
allp = [p for s in ORDER for p in FILES[s]]
def split_back(emb):
    out, i = {}, 0
    for s in ORDER:
        n = len(FILES[s]); out[s] = emb[i:i + n].astype(np.float32); i += n
    return out
for name, fn in ENCODERS.items():
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        print(name, 'already extracted'); continue
    t0 = time.time()
    try:
        emb = fn(allp); np.savez(p, **split_back(emb)); note('%-7s dim %4d | %d files in %.1f min' % (name, emb.shape[1], len(allp), (time.time() - t0) / 60))
    except Exception as e:
        note('ENCODER %s FAILED: %r' % (name, repr(e)[:300]))

In [ ]:
# Cell 5 - compare combos on top of R4 (challenge + all MAV, prototypes; 5-fold x 3 seeds; reference recomputed)
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in ENCODERS:
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        d = np.load(p); FEAT[name] = {s: d[s] for s in ORDER}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
N_CH = len(y); ext_spk = list(MAV['spk']) + list(EXTRA['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_combo(F_raw, V_raw, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j]); opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']
COMBOS = {'S0_R4_ref': (['clip'], R4), 'S1_+sam100': (['clip'], R4 + ['sam100']), 'S2_+w2vb': (['clip'], R4 + ['w2vb']), 'S3_+both': (['clip'], R4 + ['sam100', 'w2vb'])}
COMBOS = {k: v for k, v in COMBOS.items() if all(n in FEAT for n in v[0] + v[1])}
print('Combos:', list(COMBOS))
CSV = os.path.join(RESDIR, 'nb21_cv.csv'); done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold']); t0 = time.time()
for name, (fn_, vn_) in COMBOS.items():
    F_raw, V_raw = build_train(fn_), build_train(vn_)
    for seed in SEEDS:
        for k in range(K):
            if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]; model, best_ep, XF, XV, _ = train_combo(F_raw, V_raw, tr_idx, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False); done = pd.concat([done, row])
            print('%-12s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().groupby('combo').mean(); tab = tab.loc[[c for c in COMBOS if c in tab.index]].round(2)
note('MULTILINGUAL VOICE ENCODERS on top of R4 (challenge + all MAV, prototypes; 5-fold x 3 seeds)'); note(tab.to_string())
BEST = tab['overall'].idxmin(); note('BEST: %s (%.2f) vs reference %.2f' % (BEST, tab.loc[BEST, 'overall'], tab.loc['S0_R4_ref', 'overall']))

In [ ]:
# Cell 6 - ZIP for the best combo (no VoxCeleb) + saved dev scores
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
fn_, vn_ = COMBOS[BEST]; F_raw, V_raw = build_train(fn_), build_train(vn_)
ep = int(np.median(R[R.combo == BEST]['best_ep']))
finals = [train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=s, max_epochs=ep)[0] for s in SEEDS]
_, _, ch = dnorm(F_raw, V_raw, np.concatenate([np.arange(N_CH), EXT]))
sc = {}
for track in ['no_gender', 'gender']:
    for lang_ in LANG_FILE:
        split = track + '_' + lang_
        xf = torch.tensor(((build(fn_, split) - ch[0]) / ch[1]).astype(np.float32), device=dev); xv = torch.tensor(((build(vn_, split) - ch[2]) / ch[3]).astype(np.float32), device=dev)
        sc[split] = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
tag = BEST.split('_')[0]; OUT = '/content/sub_21_' + tag
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang_, fname in LANG_FILE.items():
        split = track + '_' + lang_; pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
        assert len(pair_ids) == len(sc[split]) and np.isfinite(sc[split]).all()
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, -sc[split]):
                f.write('%s %.6f\n' % (pid, v))
zp = os.path.join(SUBDIR, '21_%s_noVox.zip' % tag)
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
np.savez(os.path.join(RESDIR, 'devscores_21_%s.npz' % tag), **sc); note('Submission ZIP saved: ' + zp[len(BASE):])
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)